In [1]:
import sys
sys.path.append('/workspaces/BlizzardX')

In [2]:
from src.config.config_manager import ConfigManager
from src.ghcn_daily.ghcn_data_handler import GHCNDataHandler
from src.ghcn_daily.data_fetch import DataFetcher
from src.ghcn_daily.data_processing_old import WeatherDataProcessor
import numpy as np

In [3]:
ghcn = GHCNDataHandler()
config = ConfigManager(config_directory='/workspaces/BlizzardX/src/config')
config.load_config('settings.json')
data_fetcher = DataFetcher(config_file="settings.json",data_type='dataframe')

In [4]:
stations= ghcn.get_station_data(config.get('settings.json', 'data_sources.stations'))
inventory = ghcn.get_inventory_data(config.get('settings.json', 'data_sources.inventory'))

In [5]:
s_state_list=stations[stations['STATE']=='VT']['ID'].tolist()
s_live_list=inventory[(inventory['ID'].isin(s_state_list)) & (inventory['LASTYEAR']>2024) & (inventory['FIRSTYEAR']<2015)]['ID'].unique().tolist()

In [6]:
data= await data_fetcher.save_data(s_live_list)

Fetching Data:   0%|                                                          | 0/8 [00:00<?, ?it/s]

Fetching Data: 100%|██████████████████████████████████████████████████| 8/8 [00:11<00:00,  1.49s/it]


CPU usage is high! Decreasing workers to 4
CPU usage is high! Decreasing workers to 2
CPU usage is stable. Increasing workers to 4
CPU usage is stable. Increasing workers to 6
CPU usage is high! Decreasing workers to 4
CPU usage is high! Decreasing workers to 2
CPU usage is stable. Increasing workers to 4
CPU usage is stable. Increasing workers to 6
CPU usage is high! Decreasing workers to 4
CPU usage is stable. Increasing workers to 6
CPU usage is high! Decreasing workers to 4
CPU usage is stable. Increasing workers to 6
CPU usage is high! Decreasing workers to 4
CPU usage is stable. Increasing workers to 6
CPU usage is high! Decreasing workers to 4
CPU usage is high! Decreasing workers to 2
CPU usage is stable. Increasing workers to 4
CPU usage is stable. Increasing workers to 6
CPU usage is high! Decreasing workers to 4
CPU usage is stable. Increasing workers to 6
CPU usage is high! Decreasing workers to 4
CPU usage is stable. Increasing workers to 6
CPU usage is high! Decreasing wo

In [7]:
flag_columns = [col for col in data.columns if 'FLAG' in col]
data= data.drop(columns=flag_columns)
#data.replace(-9999.0, np.nan, inplace=True)
weather_variables = ['TMAX', 'TMIN', 'SNOW', 'SNWD', 'PRCP']

In [8]:
from src.ghcn_daily.data_processing import WeatherDataTransformer,WeatherDataImputer
processor = WeatherDataTransformer(data, weather_variables)

In [9]:
df=processor.process_data()

In [10]:
df.replace(-9999.0, np.nan, inplace=True)
df.replace(-999.9, np.nan, inplace=True)

In [11]:
df.head()

,DATE,ID,TMAX,TMIN,SNOW,SNWD,PRCP,Season
0,2009-04-01,US1VTAD0005,NaN,NaN,NaN,NaN,NaN,Spring
1,2009-04-02,US1VTAD0005,NaN,NaN,NaN,NaN,NaN,Spring
2,2009-04-03,US1VTAD0005,NaN,NaN,NaN,NaN,NaN,Spring
3,2009-04-04,US1VTAD0005,NaN,NaN,NaN,NaN,NaN,Spring
4,2009-04-05,US1VTAD0005,NaN,NaN,NaN,NaN,NaN,Spring


In [12]:
import pandas as pd
df = pd.merge(df, stations, on='ID', how='left')
df = df[['DATE','ID', 'LATITUDE', 'LONGITUDE', 'ELEVATION', 'NAME', 'Season', 'TMIN', 'TMAX', 'PRCP', 'SNOW', 'SNWD']]

In [13]:
from src.ghcn_daily.data_filtering import WeatherDataFilter
filter = WeatherDataFilter(df)

In [14]:
df=df[df['ID'].isin(filter.get_stations_with_zero_missing_dates())]

In [15]:
df=df[df['ID'].isin(filter.get_stations_with_low_missing_values(threshold=5))]

In [16]:
import os
os.makedirs(os.path.dirname('/workspaces/BlizzardX/Data/processed_data.csv'), exist_ok=True)

In [17]:
imputer= WeatherDataImputer(df)

In [18]:
df=imputer.clean_all()

In [19]:
df.to_csv('/workspaces/BlizzardX/Data/Diff_state.csv', index=False)

In [55]:
df.head(50)

,DATE,Station_ID,LATITUDE,LONGITUDE,ELEVATION,NAME,Season,TMIN,TMAX,PRCP,...,Snowfall_Intensity,SNWD_Snowfall_Diff,PRCP_Lag1,PRCP_Lag2,Cumulative_Precipitation_7,Rolling_Sum_PRCP_14,TMAX_PRCP_Interaction,TMIN_SNOW_Interaction,PRCP_SNOW_Interaction,Cold_Event
175022,2011-11-01,USC00430193,44.9989,-71.7097,514.2,VT AVERILL,Fall,-7.20,6.70,0.00,...,0.00,0.00,0.00,0.00,0.00,0.00,0.00,-0.00,0.00,0
175023,2011-11-02,USC00430193,44.9989,-71.7097,514.2,VT AVERILL,Fall,-3.90,9.40,0.00,...,0.00,0.00,0.00,0.00,0.00,0.00,0.00,-0.00,0.00,0
175024,2011-11-03,USC00430193,44.9989,-71.7097,514.2,VT AVERILL,Fall,-2.20,11.70,0.00,...,0.00,0.00,0.00,0.00,0.00,0.00,0.00,-0.00,0.00,0
175025,2011-11-04,USC00430193,44.9989,-71.7097,514.2,VT AVERILL,Fall,-1.70,12.80,0.00,...,0.00,0.00,0.00,0.00,0.00,0.00,0.00,-0.00,0.00,0
175026,2011-11-05,USC00430193,44.9989,-71.7097,514.2,VT AVERILL,Fall,-5.00,1.70,0.30,...,3.00,-3.00,0.00,0.00,0.30,0.30,0.51,-15.00,0.90,0
175027,2011-11-06,USC00430193,44.9989,-71.7097,514.2,VT AVERILL,Fall,-5.85,6.95,0.15,...,1.50,-1.50,0.30,0.00,0.45,0.45,1.04,-8.77,0.22,0
175028,2011-11-07,USC00430193,44.9989,-71.7097,514.2,VT AVERILL,Fall,-6.70,12.20,0.00,...,0.00,0.00,0.15,0.30,0.45,0.45,0.00,-0.00,0.00,0
175029,2011-11-08,USC00430193,44.9989,-71.7097,514.2,VT AVERILL,Fall,-6.70,13.90,0.00,...,0.00,0.00,0.00,0.15,0.45,0.45,0.00,-0.00,0.00,0
175030,2011-11-09,USC00430193,44.9989,-71.7097,514.2,VT AVERILL,Fall,4.40,13.90,0.00,...,0.00,0.00,0.00,0.00,0.45,0.45,0.00,0.00,0.00,0
175031,2011-11-10,USC00430193,44.9989,-71.7097,514.2,VT AVERILL,Fall,6.70,18.30,0.00,...,0.00,0.00,0.00,0.00,0.45,0.45,0.00,0.00,0.00,0


In [21]:
df.columns
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 44282 entries, 175022 to 650300
Data columns (total 12 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   DATE       44282 non-null  datetime64[ns]
 1   ID         44282 non-null  object        
 2   LATITUDE   44282 non-null  float64       
 3   LONGITUDE  44282 non-null  float64       
 4   ELEVATION  44282 non-null  float64       
 5   NAME       44282 non-null  object        
 6   Season     44282 non-null  object        
 7   TMIN       44282 non-null  float64       
 8   TMAX       44282 non-null  float64       
 9   PRCP       44282 non-null  float64       
 10  SNOW       44282 non-null  float64       
 11  SNWD       44282 non-null  float64       
dtypes: datetime64[ns](1), float64(8), object(3)
memory usage: 5.4+ MB


In [5]:
import sys
sys.path.append('/workspaces/BlizzardX')

In [1]:
import pandas as pd

# Load Vermont Data
vermont_data = pd.read_csv('/workspaces/BlizzardX/Data/Diff_state.csv')

# Display basic info
print(f"✅ Vermont Data Shape: {vermont_data.shape}")
print(f"✅ Vermont Data Columns: {vermont_data.columns.tolist()}")
print(vermont_data.head())


✅ Vermont Data Shape: (44282, 12)
✅ Vermont Data Columns: ['DATE', 'ID', 'LATITUDE', 'LONGITUDE', 'ELEVATION', 'NAME', 'Season', 'TMIN', 'TMAX', 'PRCP', 'SNOW', 'SNWD']
         DATE           ID  LATITUDE  LONGITUDE  ELEVATION        NAME Season  \
0  2011-11-01  USC00430193   44.9989   -71.7097      514.2  VT AVERILL   Fall   
1  2011-11-02  USC00430193   44.9989   -71.7097      514.2  VT AVERILL   Fall   
2  2011-11-03  USC00430193   44.9989   -71.7097      514.2  VT AVERILL   Fall   
3  2011-11-04  USC00430193   44.9989   -71.7097      514.2  VT AVERILL   Fall   
4  2011-11-05  USC00430193   44.9989   -71.7097      514.2  VT AVERILL   Fall   

   TMIN  TMAX  PRCP  SNOW  SNWD  
0  -7.2   6.7   0.0   0.0   0.0  
1  -3.9   9.4   0.0   0.0   0.0  
2  -2.2  11.7   0.0   0.0   0.0  
3  -1.7  12.8   0.0   0.0   0.0  
4  -5.0   1.7   0.3   3.0   0.0  


In [2]:
# 1. Rename ID to Station_ID
vermont_data = vermont_data.rename(columns={'ID': 'Station_ID'})

# 2. Convert DATE to datetime
vermont_data['DATE'] = pd.to_datetime(vermont_data['DATE'])

# 3. Print confirmation
print(f"✅ Vermont data ready for Feature Engineering: {vermont_data.shape}")
print(vermont_data[['DATE', 'Station_ID', 'TMIN', 'TMAX', 'SNOW', 'SNWD']].head())


✅ Vermont data ready for Feature Engineering: (44282, 12)
        DATE   Station_ID  TMIN  TMAX  SNOW  SNWD
0 2011-11-01  USC00430193  -7.2   6.7   0.0   0.0
1 2011-11-02  USC00430193  -3.9   9.4   0.0   0.0
2 2011-11-03  USC00430193  -2.2  11.7   0.0   0.0
3 2011-11-04  USC00430193  -1.7  12.8   0.0   0.0
4 2011-11-05  USC00430193  -5.0   1.7   3.0   0.0


In [6]:
import os
from src.Model.feature_engineering import FeatureEngineering

# Initialize Feature Engineering
fe_vermont = FeatureEngineering(vermont_data)

# Apply all features (rolling averages, lags, cumulative features, interactions)
vermont_features_df = fe_vermont.apply_all_features()

print(f"✅ Vermont Feature Engineered Data Shape: {vermont_features_df.shape}")
print(vermont_features_df.head())


✅ Vermont Feature Engineered Data Shape: (44282, 45)
        DATE   Station_ID  LATITUDE  LONGITUDE  ELEVATION        NAME Season  \
0 2011-11-01  USC00430193   44.9989   -71.7097      514.2  VT AVERILL   Fall   
1 2011-11-02  USC00430193   44.9989   -71.7097      514.2  VT AVERILL   Fall   
2 2011-11-03  USC00430193   44.9989   -71.7097      514.2  VT AVERILL   Fall   
3 2011-11-04  USC00430193   44.9989   -71.7097      514.2  VT AVERILL   Fall   
4 2011-11-05  USC00430193   44.9989   -71.7097      514.2  VT AVERILL   Fall   

   TMIN  TMAX  PRCP  ...  SNWD_TMIN_Interaction  Snowfall_Intensity  \
0  -7.2   6.7   0.0  ...                   -0.0                 0.0   
1  -3.9   9.4   0.0  ...                   -0.0                 0.0   
2  -2.2  11.7   0.0  ...                   -0.0                 0.0   
3  -1.7  12.8   0.0  ...                   -0.0                 0.0   
4  -5.0   1.7   0.3  ...                   -0.0                 3.0   

  SNWD_Snowfall_Diff  PRCP_Lag1  PRCP_L

In [8]:
# 1. Prepare feature columns (drop DATE, Station_ID, NAME, TMIN)
non_feature_cols = ['DATE', 'Station_ID', 'NAME', 'Season', 'TMIN']

# 2. Select feature columns
feature_cols = [col for col in vermont_features_df.columns if col not in non_feature_cols]

# 3. Prepare X for prediction
X_vermont_future = vermont_features_df[feature_cols].copy()

# 4. Add dummy Cold_Event column if needed
X_vermont_future['Cold_Event'] = 0

print(f"✅ Vermont X_future ready for prediction: {X_vermont_future.shape}")


✅ Vermont X_future ready for prediction: (44282, 41)


In [12]:
# Double-check and remove any non-numeric columns

# Drop columns that are non-numeric manually (safe)
X_vermont_future = X_vermont_future.drop(columns=[col for col in X_vermont_future.columns if X_vermont_future[col].dtype == 'object'])

print(f"✅ X_vermont_future cleaned, new shape: {X_vermont_future.shape}")
print(X_vermont_future.dtypes)  # Should now be all int, float, bool


✅ X_vermont_future cleaned, new shape: (44282, 40)
LATITUDE                         float64
LONGITUDE                        float64
ELEVATION                        float64
TMAX                             float64
PRCP                             float64
SNOW                             float64
SNWD                             float64
Station_Lat_Long_Interaction     float64
Day_of_Week                        int32
Day_of_Year                        int32
Month                              int32
Temp_Diff                        float64
Rolling_Mean_TMIN_7              float64
Rolling_10thPercentile_TMIN_7    float64
Rolling_Mean_TMIN_30             float64
Rolling_Max_TMIN_30              float64
Rolling_Min_TMIN_30              float64
TMIN_Rolling_30_Diff             float64
EWMA_TMIN_7                      float64
EWMA_TMIN_30                     float64
Seasonal_TMIN_Anomaly            float64
TMIN_Lag1                        float64
SnowyDay                           int64
SnowyD

In [9]:
import joblib
import os

os.makedirs("models", exist_ok=True)

In [10]:
tmin_model = joblib.load('/workspaces/BlizzardX/Notebooks/models/xgboost_tmin_feature_model.pkl')

In [14]:
# Re-add Season column to X_vermont_future from the feature-engineered Vermont dataframe
X_vermont_future['Season'] = vermont_features_df['Season']

# Map Season text to numeric codes
season_mapping = {'Winter': 0, 'Spring': 1, 'Summer': 2, 'Fall': 3}
X_vermont_future['Season'] = X_vermont_future['Season'].map(season_mapping)

print(f"✅ Season column added and mapped correctly! New shape: {X_vermont_future.shape}")
print(X_vermont_future[['Season']].head())

✅ Season column added and mapped correctly! New shape: (44282, 41)
   Season
0       3
1       3
2       3
3       3
4       3


In [16]:
# Correct feature order that model expects
correct_feature_order = [
    'LATITUDE', 'LONGITUDE', 'ELEVATION', 'Season',
    'TMAX', 'PRCP', 'SNOW', 'SNWD', 'Station_Lat_Long_Interaction',
    'Day_of_Week', 'Day_of_Year', 'Month', 'Temp_Diff',
    'Rolling_Mean_TMIN_7', 'Rolling_10thPercentile_TMIN_7',
    'Rolling_Mean_TMIN_30', 'Rolling_Max_TMIN_30', 'Rolling_Min_TMIN_30',
    'TMIN_Rolling_30_Diff', 'EWMA_TMIN_7', 'EWMA_TMIN_30',
    'Seasonal_TMIN_Anomaly', 'TMIN_Lag1', 'SnowyDay', 'SnowyDaysCount_7',
    'Cumulative_SnowDepth_7', 'Rolling_Sum_SNWD_7', 'SNWD_Lag1', 'SNWD_Lag2',
    'Cumulative_Snowfall_Lag7', 'SNWD_TMIN_Interaction', 'Snowfall_Intensity',
    'SNWD_Snowfall_Diff', 'PRCP_Lag1', 'PRCP_Lag2',
    'Cumulative_Precipitation_7', 'Rolling_Sum_PRCP_14',
    'TMAX_PRCP_Interaction', 'TMIN_SNOW_Interaction', 'PRCP_SNOW_Interaction',
    'Cold_Event'
]

# Reorder X_vermont_future
X_vermont_future = X_vermont_future[correct_feature_order]

print(f"✅ X_vermont_future columns reordered to match model training!")
print(X_vermont_future.columns.tolist())


✅ X_vermont_future columns reordered to match model training!
['LATITUDE', 'LONGITUDE', 'ELEVATION', 'Season', 'TMAX', 'PRCP', 'SNOW', 'SNWD', 'Station_Lat_Long_Interaction', 'Day_of_Week', 'Day_of_Year', 'Month', 'Temp_Diff', 'Rolling_Mean_TMIN_7', 'Rolling_10thPercentile_TMIN_7', 'Rolling_Mean_TMIN_30', 'Rolling_Max_TMIN_30', 'Rolling_Min_TMIN_30', 'TMIN_Rolling_30_Diff', 'EWMA_TMIN_7', 'EWMA_TMIN_30', 'Seasonal_TMIN_Anomaly', 'TMIN_Lag1', 'SnowyDay', 'SnowyDaysCount_7', 'Cumulative_SnowDepth_7', 'Rolling_Sum_SNWD_7', 'SNWD_Lag1', 'SNWD_Lag2', 'Cumulative_Snowfall_Lag7', 'SNWD_TMIN_Interaction', 'Snowfall_Intensity', 'SNWD_Snowfall_Diff', 'PRCP_Lag1', 'PRCP_Lag2', 'Cumulative_Precipitation_7', 'Rolling_Sum_PRCP_14', 'TMAX_PRCP_Interaction', 'TMIN_SNOW_Interaction', 'PRCP_SNOW_Interaction', 'Cold_Event']


In [17]:
# Predict Vermont TMIN now
predicted_vermont_tmin = tmin_model.predict(X_vermont_future)

# Attach predictions back to vermont_features_df
vermont_features_df['Predicted_TMIN'] = predicted_vermont_tmin

print(f"✅ Predicted Vermont TMIN successfully added!")
print(vermont_features_df[['DATE', 'Station_ID', 'Predicted_TMIN']].head())


✅ Predicted Vermont TMIN successfully added!
        DATE   Station_ID  Predicted_TMIN
0 2011-11-01  USC00430193       -7.149608
1 2011-11-02  USC00430193       -4.226179
2 2011-11-03  USC00430193       -2.477651
3 2011-11-04  USC00430193       -1.647657
4 2011-11-05  USC00430193       -5.082384


In [18]:
# Find 10th percentile cutoff for predicted TMIN
predicted_vermont_10th = vermont_features_df['Predicted_TMIN'].quantile(0.10)

print(f"✅ 10th percentile cutoff for Predicted Vermont TMIN: {predicted_vermont_10th:.2f}°C")

# Create Cold Event column
vermont_features_df['Cold_Event'] = (vermont_features_df['Predicted_TMIN'] <= predicted_vermont_10th).astype(int)

print(vermont_features_df[['DATE', 'Station_ID', 'Predicted_TMIN', 'Cold_Event']].head())


✅ 10th percentile cutoff for Predicted Vermont TMIN: -16.31°C
        DATE   Station_ID  Predicted_TMIN  Cold_Event
0 2011-11-01  USC00430193       -7.149608           0
1 2011-11-02  USC00430193       -4.226179           0
2 2011-11-03  USC00430193       -2.477651           0
3 2011-11-04  USC00430193       -1.647657           0
4 2011-11-05  USC00430193       -5.082384           0


In [19]:
# Merge predicted TMIN with actual TMIN
merged_vermont_df = vermont_features_df[['DATE', 'Station_ID', 'Predicted_TMIN', 'Cold_Event']].copy()

# Add actual TMIN back from raw Vermont dataset
merged_vermont_df = merged_vermont_df.merge(
    vermont_data[['DATE', 'Station_ID', 'TMIN']],
    on=['DATE', 'Station_ID'],
    how='inner'
)

# Rename for clarity
merged_vermont_df = merged_vermont_df.rename(columns={'TMIN': 'Actual_TMIN'})

print(f"✅ Merged Vermont dataframe ready: {merged_vermont_df.shape}")
print(merged_vermont_df.head())

✅ Merged Vermont dataframe ready: (44282, 5)
        DATE   Station_ID  Predicted_TMIN  Cold_Event  Actual_TMIN
0 2011-11-01  USC00430193       -7.149608           0         -7.2
1 2011-11-02  USC00430193       -4.226179           0         -3.9
2 2011-11-03  USC00430193       -2.477651           0         -2.2
3 2011-11-04  USC00430193       -1.647657           0         -1.7
4 2011-11-05  USC00430193       -5.082384           0         -5.0


In [20]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, accuracy_score, precision_score, recall_score, f1_score
import numpy as np

# Regression Metrics (TMIN Prediction)
mae_vermont = mean_absolute_error(merged_vermont_df['Actual_TMIN'], merged_vermont_df['Predicted_TMIN'])
rmse_vermont = np.sqrt(mean_squared_error(merged_vermont_df['Actual_TMIN'], merged_vermont_df['Predicted_TMIN']))

print(f"\n✅ Vermont TMIN Prediction Evaluation:")
print(f"MAE: {mae_vermont:.2f} °C")
print(f"RMSE: {rmse_vermont:.2f} °C")

# Classification Metrics (Cold Event Detection)
# Detect actual cold events based on actual TMIN 10th percentile
actual_vermont_10th = merged_vermont_df['Actual_TMIN'].quantile(0.10)
merged_vermont_df['Actual_Cold_Event'] = (merged_vermont_df['Actual_TMIN'] <= actual_vermont_10th).astype(int)

print(f"\n✅ 10th percentile cutoff for Actual Vermont TMIN: {actual_vermont_10th:.2f}°C")

# Cold Event Prediction Metrics
accuracy_vermont = accuracy_score(merged_vermont_df['Actual_Cold_Event'], merged_vermont_df['Cold_Event'])
precision_vermont = precision_score(merged_vermont_df['Actual_Cold_Event'], merged_vermont_df['Cold_Event'])
recall_vermont = recall_score(merged_vermont_df['Actual_Cold_Event'], merged_vermont_df['Cold_Event'])
f1_vermont = f1_score(merged_vermont_df['Actual_Cold_Event'], merged_vermont_df['Cold_Event'])

print(f"\n✅ Vermont Cold Event Prediction Evaluation:")
print(f"Accuracy: {accuracy_vermont:.2%}")
print(f"Precision: {precision_vermont:.2%}")
print(f"Recall: {recall_vermont:.2%}")
print(f"F1 Score: {f1_vermont:.2%}")


✅ Vermont TMIN Prediction Evaluation:
MAE: 0.38 °C
RMSE: 0.53 °C

✅ 10th percentile cutoff for Actual Vermont TMIN: -16.10°C

✅ Vermont Cold Event Prediction Evaluation:
Accuracy: 99.26%
Precision: 98.33%
Recall: 94.47%
F1 Score: 96.36%


# 🏔️ Vermont Model Performance Evaluation

## 1. Objective
- Evaluate how well the model generalizes to a new unseen region (Vermont stations).

## 2. TMIN Prediction Results
- **Mean Absolute Error (MAE)**: **0.38°C**
- **Root Mean Squared Error (RMSE)**: **0.53°C**

## 3. Cold Event Detection Results
- **10th percentile cutoff for Actual Vermont TMIN**: **-16.10°C**
- **Accuracy**: **99.26%**
- **Precision**: **98.33%**
- **Recall**: **94.47%**
- **F1 Score**: **96.36%**

## 4. Observations
- The model maintains extremely low prediction error even in a new region.
- Cold event detection remains highly accurate and balanced.
- This proves strong generalization capability beyond New Hampshire, supporting wider deployment.

---


### Getting Tmin for specific dates 

In [23]:
mask = (vermont_features_df['DATE'] >= '2025-01-01') & (vermont_features_df['DATE'] <= '2025-02-28')
jan_feb_2025_df = vermont_features_df.loc[mask]

In [24]:
print(jan_feb_2025_df[['DATE', 'Station_ID', 'Predicted_TMIN']])

            DATE   Station_ID  Predicted_TMIN
4810  2025-01-01  USC00430193       -2.264987
4811  2025-01-02  USC00430193       -7.173978
4812  2025-01-03  USC00430193       -9.749469
4813  2025-01-04  USC00430193      -16.818995
4814  2025-01-05  USC00430193      -21.457109
...          ...          ...             ...
44206 2025-02-24  USC00439988       -6.081662
44207 2025-02-25  USC00439988       -5.567606
44208 2025-02-26  USC00439988        1.094428
44209 2025-02-27  USC00439988       -4.486825
44210 2025-02-28  USC00439988       -3.067810

[354 rows x 3 columns]


In [25]:
# 1. Ensure DATE column is datetime (if not already)
merged_vermont_df['DATE'] = pd.to_datetime(merged_vermont_df['DATE'])

# 2. Filter between 2025-01-01 and 2025-02-28
mask = (merged_vermont_df['DATE'] >= '2025-01-01') & (merged_vermont_df['DATE'] <= '2025-02-28')
jan_feb_vermont_df = merged_vermont_df.loc[mask]

# 3. Display Date, Station_ID, Predicted_TMIN, Actual_TMIN
print(jan_feb_vermont_df[['DATE', 'Station_ID', 'Predicted_TMIN', 'Actual_TMIN']])


            DATE   Station_ID  Predicted_TMIN  Actual_TMIN
4810  2025-01-01  USC00430193       -2.264987         -2.2
4811  2025-01-02  USC00430193       -7.173978         -7.2
4812  2025-01-03  USC00430193       -9.749469        -10.0
4813  2025-01-04  USC00430193      -16.818995        -17.2
4814  2025-01-05  USC00430193      -21.457109        -22.2
...          ...          ...             ...          ...
44206 2025-02-24  USC00439988       -6.081662         -5.6
44207 2025-02-25  USC00439988       -5.567606         -5.0
44208 2025-02-26  USC00439988        1.094428          1.1
44209 2025-02-27  USC00439988       -4.486825         -4.4
44210 2025-02-28  USC00439988       -3.067810         -2.8

[354 rows x 4 columns]


In [26]:
# Save Vermont Jan-Feb 2025 data
jan_feb_vermont_df[['DATE', 'Station_ID', 'Predicted_TMIN', 'Actual_TMIN']].to_csv('Vermont_Predicted_Actual_JanFeb2025.csv', index=False)

print("✅ Vermont Jan-Feb 2025 Predicted vs Actual TMIN saved to Vermont_Predicted_Actual_JanFeb2025.csv")

✅ Vermont Jan-Feb 2025 Predicted vs Actual TMIN saved to Vermont_Predicted_Actual_JanFeb2025.csv
